# BIND2 Methods Paper — Figures (main)

**One model**, mass + thermodynamics as a function of redshift and parameters (`fm_redshift` @ z=0), validated by **halo-mass bin** in the **trained regime** ($M_{200c}\ge 10^{13}$, the training cut). Every figure loads a precomputed cache built by `tools/paper_cache/` — no compute here. Build the cache with `bash run_paper_cache.sh` (see the repo README of that dir).

Sections: §1 showcase · §2 integrated mass + parameter response · §3 profiles + parameter response · §4 field-level (1P) · §5 power spectrum · §6 halo shapes + parameter response.

In [ ]:

import sys
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2/tools/paper_cache')
import os, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import paper_config as C

CACHE = C.CACHE_DIR
def L(name):
    p = CACHE / name
    if name.endswith('.pkl'):
        return pickle.load(open(p, 'rb'))
    return dict(np.load(p, allow_pickle=False))

FIG_DIR = Path('paper_figures'); FIG_DIR.mkdir(exist_ok=True)
def save_fig(fig, name, ext=('pdf', 'png')):
    for e in ext:
        fig.savefig(FIG_DIR / f'{name}.{e}', dpi=300, bbox_inches='tight')
    print('  saved', name)

try:
    import scienceplots  # noqa: F401
    plt.style.use(['science', 'notebook'])
except Exception:
    pass
# plt.rcParams.update({'font.size': 10, 'font.family': 'serif', 'mathtext.fontset': 'cm',
#                      'figure.dpi': 110, 'savefig.dpi': 300, 'axes.grid': False})

SUITE_COLORS = C.SUITE_COLORS; SUITE_DISPLAY = C.SUITE_DISPLAY
MASS_CH = C.MASS_CHANNELS; CH_DISPLAY = C.CH_DISPLAY
BIN_LABELS = C.MASS_BIN_LABELS; N_BINS = C.N_MASS_BINS
PARAM_LABELS = C.PARAM_LABELS
# Trained-regime restriction: the paper only uses halos with M200c >= 1e13 (the
# training cut). Lower bins exist in the cache but are never plotted.
MIN_LOG_M200 = 13.0
BINS = [b for b in range(N_BINS) if C.MASS_EDGES[b] >= MIN_LOG_M200]
BIN_CMAP = np.zeros((N_BINS, 4))
BIN_CMAP[BINS] = plt.cm.viridis(np.linspace(0.1, 0.9, len(BINS)))
print('Cache :', CACHE)
print('Model :', C.MODEL_TAG, '| suites', {s: 0 for s in C.SUITES})
print('Files :', sorted(p.name for p in CACHE.glob("*.pkl")) + sorted(p.name for p in CACHE.glob("*.npz")))


## §1 · Full-box showcase (Fig 1)

DMO → BIND2 mass fields vs hydro truth for one CV box. The BIND row shows the taper-free pasted patches (`hydro_canvas`).

In [ ]:

rec = C.discover_sims(('CV',))[0]
fm = C.load_full_maps(rec); comp = C.load_composite(rec)
dmo, truth = fm['dmo_fullbox'], fm['truth_maps']
canvas = comp['hydro_canvas']     # patches-on-black; drives the clean residual
composite = comp['composite']     # blended full-box map

def showcase(bind_field, tag):
    fig, axs = plt.subplots(ncols=4, nrows=2, figsize=(16, 8), sharex=True, sharey=True, gridspec_kw={'wspace': 0.005, 'hspace': 0.05})
    def _im(ax, img, title=None, vmax=None):
        pos = img[img > 0]; vmax = vmax or np.quantile(pos, 0.999); vmin = max(np.quantile(pos, 0.05), 1e-8)
        ax.imshow(img, origin='lower', cmap='magma', norm=SymLogNorm(linthresh=max(vmin,1e-6), vmin=0, vmax=vmax, base=10), extent=[0, 50, 0, 50])
        if title: ax.set_title(title)

    vmax = []
    for c in range(3):
        pos = np.concatenate([truth[c].ravel(), bind_field[c].ravel()]); vmax.append(np.quantile(pos[pos>0], 0.999))
    _im(axs[0,0], dmo, 'DMO (Nbody)')
    for c in range(3): _im(axs[0,c+1], truth[c], CH_DISPLAY[MASS_CH[c]], vmax[c])
    _im(axs[1,0], dmo)
    for c in range(3): _im(axs[1,c+1], bind_field[c], vmax=vmax[c])

    # X label only on bottom row
    for ax in axs[1]: ax.set_xlabel('X [Mpc/h]')
    # Y label only on leftmost column
    axs[0,0].set_ylabel('Y [Mpc/h]'); axs[1,0].set_ylabel('Y [Mpc/h]')

    # Row super-labels
    fig.text(0.01, 0.70, 'Truth', va='center', ha='center', fontsize=20, fontweight='bold', rotation='vertical')
    fig.text(0.01, 0.30, 'BIND',  va='center', ha='center', fontsize=20, fontweight='bold', rotation='vertical')

    fig.subplots_adjust(left=0.06)
    plt.tight_layout()
    save_fig(fig, tag); plt.show()

showcase(canvas, 'fig1_showcase')                # v1: BIND row = pasted patches (matches original)


## §2 · Integrated mass + parameter response

Fig 2 consolidated gen-vs-truth scatter + mass distribution **per mass bin** (R200 aperture); KS/median-bias table; Fig 3a parameter → mass Spearman (`mass_param.pkl`); Fig 3a-bis dominant correlations vs bin.

In [ ]:

tbl = L('mass_table.pkl')
tbl = tbl[tbl.mass_bin.isin(BINS)]   # trained regime only (>=1e13)
# Fig 2 — consolidated: gen vs truth scatter (left) + mass distribution (right), per mass bin, R200 aperture

# Pre-compute global x range for scatter (left col) across all bins and channels
_all_sc = []
for b in BINS:
    sub = tbl[tbl.mass_bin == b]
    for ch in MASS_CH:
        t = sub[f'truth_{ch}_rvir']; g = sub[f'gen_{ch}_rvir']; m = (t > 0) & (g > 0)
        if m.sum() > 0: _all_sc.extend([np.log10(t[m]).values, np.log10(g[m]).values])
sc_lo = np.concatenate(_all_sc).min() - 0.15
sc_hi = np.concatenate(_all_sc).max() + 0.15

fig, axes = plt.subplots(len(BINS), 2, figsize=(10, 3*len(BINS)), constrained_layout=True,
                         sharex='col')

for row, b in enumerate(BINS):
    sub = tbl[tbl.mass_bin == b]
    ax_sc, ax_hi = axes[row, 0], axes[row, 1]

    for c, ch in enumerate(MASS_CH):
        t = sub[f'truth_{ch}_rvir']; g = sub[f'gen_{ch}_rvir']
        m = (t > 0) & (g > 0); xt = np.log10(t[m]); xg = np.log10(g[m])
        ax_sc.scatter(xt, xg, s=6, alpha=0.15, color=f'C{c}', rasterized=True,
                      label=CH_DISPLAY[ch] if row == 0 else None)
        # bin medians with errorbars
        bins_s = np.linspace(xt.min(), xt.max(), 12)
        bc = 0.5 * (bins_s[:-1] + bins_s[1:]); idx = np.digitize(xt, bins_s) - 1
        med = [np.median(xg[idx==i]) if (idx==i).sum() > 3 else np.nan for i in range(len(bc))]
        sd  = [np.std(xg[idx==i])    if (idx==i).sum() > 3 else np.nan for i in range(len(bc))]
        ax_sc.errorbar(bc, med, yerr=sd, fmt='o', ms=4, color=f'C{c}',
                       mec='k', ecolor='k', capsize=2, zorder=5, lw=0.8)
        bins_h = np.linspace(min(xt.min(), xg.min()), max(xt.max(), xg.max()), 25)
        ax_hi.hist(xt, bins=bins_h, density=True, histtype='step', lw=1.5, color=f'C{c}')
        ax_hi.hist(xg, bins=bins_h, density=True, histtype='stepfilled', alpha=0.35, color=f'C{c}')

    ax_sc.plot([sc_lo, sc_hi], [sc_lo, sc_hi], 'k--', lw=0.8)
    ax_sc.set_ylim(sc_lo, sc_hi)   # y per-panel; x shared via sharex='col'

    ax_sc.set_ylabel(BIN_LABELS[b] + '\n' + r'$\log_{10} M_{\rm BIND}$')
    ax_hi.set_ylabel(r'$\rho$')
    if row < len(BINS) - 1:
        ax_sc.tick_params(labelbottom=False); ax_hi.tick_params(labelbottom=False)
    else:
        ax_sc.set_xlabel(r'$\log_{10} M_{\rm truth}\ [r \leq R_{200}]$')
        ax_hi.set_xlabel(r'$\log_{10} M\ [r \leq R_{200}]$')

axes[0, 0].set_xlim(sc_lo, sc_hi)   # set once; propagates to all left panels via sharex
axes[0, 0].set_title(r'$M_{\rm BIND}$ vs $M_{\rm truth}$')
axes[0, 1].set_title('Mass distribution')
axes[0, 0].legend(fontsize=12)
hist_handles = [Line2D([0],[0], color='gray', lw=1.5, label='Truth'),
                Patch(facecolor='gray', alpha=0.35, label='BIND')]
axes[0, 1].legend(handles=hist_handles, fontsize=12)

save_fig(fig, 'fig2_mass_by_bin'); plt.show()


In [ ]:

# Quantitative distribution comparison: KS test + median log-bias, per channel × mass bin
from scipy.stats import ks_2samp

ks_stat  = np.full((len(BINS), len(MASS_CH)), np.nan)
ks_pval  = np.full((len(BINS), len(MASS_CH)), np.nan)
log_bias = np.full((len(BINS), len(MASS_CH)), np.nan)   # median(log M_BIND - log M_truth)

for row, b in enumerate(BINS):
    sub = tbl[tbl.mass_bin == b]
    for c, ch in enumerate(MASS_CH):
        t = sub[f'truth_{ch}_rvir']; g = sub[f'gen_{ch}_rvir']
        m = (t > 0) & (g > 0)
        if m.sum() < 10: continue
        xt, xg = np.log10(t[m].values), np.log10(g[m].values)
        st, pv = ks_2samp(xt, xg)
        ks_stat[row, c]  = st
        ks_pval[row, c]  = pv
        log_bias[row, c] = np.median(xg - xt)

ch_labels = [CH_DISPLAY[ch] for ch in MASS_CH]
bin_labels = [BIN_LABELS[b] for b in BINS]
fig, axes = plt.subplots(1, 2, figsize=(10, 0.6*len(BINS) + 2.5), constrained_layout=True)

for ax, data, title, fmt, cmap, vmin, vmax in [
    (axes[0], log_bias, r'Median $\log_{10}(M_{\rm BIND}/M_{\rm truth})$',
     '{:+.3f}', 'RdBu_r', -0.3, 0.3),
    (axes[1], np.log10(np.clip(ks_pval, 1e-10, 1)),
     r'KS $\log_{10}(p)$  [dashed = $p=0.05$]',
     '{:.1f}', 'RdYlGn', -6, 0),
]:
    im = ax.imshow(data, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax, origin='upper')
    for row in range(len(BINS)):
        for c in range(len(MASS_CH)):
            v = data[row, c]
            if np.isfinite(v):
                ax.text(c, row, fmt.format(v), ha='center', va='center', fontsize=9,
                        color='white' if abs(v) > 0.6*(vmax-vmin)/2 + vmin else 'k')
    ax.set_xticks(range(len(MASS_CH))); ax.set_xticklabels(ch_labels)
    ax.set_yticks(range(len(BINS))); ax.set_yticklabels(bin_labels)
    ax.set_ylabel(r'$\log_{10} M_{200c}$ bin'); ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.85)

# mark KS significance threshold (p=0.05 → log p ≈ -1.3)
axes[1].axvline(-1, color='k', ls='--', lw=0.5)   # visual guide only; significance is per-cell

save_fig(fig, 'fig2_mass_ks_bias'); plt.show()

print("\nMedian log-bias  (rows=bins, cols=channels):")
print(pd.DataFrame(log_bias.round(3), index=bin_labels, columns=ch_labels).to_string())
print("\nKS p-value:")
print(pd.DataFrame(ks_pval.round(4), index=bin_labels, columns=ch_labels).to_string())


In [ ]:

# Fig 3a — parameter -> mass Spearman (True / BIND / residual)
mp = L('mass_param.pkl'); rho = mp['rho_mass']['trained']; chans = mp['channels']
astro = [j for j in range(C.N_PARAMS) if (j+1) not in C.COSMO_PARAM_IDX]
order = sorted(astro)   # all astro params, in index order (matches the original layout)
xl = [PARAM_LABELS[j+1] for j in order]
fig, axes = plt.subplots(len(chans), 1, figsize=(13, 1.7*len(chans)), sharex=True)
for ci,(ch,ax) in enumerate(zip(chans, axes)):
    d = np.vstack([rho['True'][ci,order], rho['BIND'][ci,order], rho['True'][ci,order]-rho['BIND'][ci,order]])
    im = ax.imshow(d, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    for r in range(3):
        for k in range(len(order)):
            if np.isfinite(d[r,k]): ax.text(k,r,f'{d[r,k]:.2f}',ha='center',va='center',fontsize=7,color='white' if abs(d[r,k])>0.4 else 'k')
    ax.axhline(1.5, color='k', lw=1, ls='--'); ax.set_yticks([0,1,2]); ax.set_yticklabels(['True','BIND',r'$\Delta$']); ax.set_ylabel(CH_DISPLAY.get(ch,ch))
axes[-1].set_xticks(range(len(order))); axes[-1].set_xticklabels(xl, rotation=45, ha='right')
fig.colorbar(im, ax=axes, fraction=0.12).set_label(r'$\rho_S$')
save_fig(fig, 'fig3a_spearman_param_mass'); plt.show()


In [ ]:

# Fig 3a-bis — how dominant parameter correlations shift with mass bin
from scipy.stats import spearmanr

# Merge per-sim params from mass_param.pkl onto tbl (keyed by suite + sim_id)
# sim_table has columns p1..p35; rename to param_0..param_34
if 'param_0' not in tbl.columns:
    _mp = L('mass_param.pkl')
    _st = _mp['sim_table'].copy()
    _st = _st.rename(columns={f'p{j+1}': f'param_{j}' for j in range(C.N_PARAMS)})
    _key = ['suite', 'sim_id']
    tbl = tbl.merge(_st[_key + [f'param_{j}' for j in range(C.N_PARAMS)]], on=_key, how='left')

param_cols = sorted([c for c in tbl.columns if c.startswith('param_')],
                    key=lambda c: int(c.split('_')[1]))
if not param_cols:
    print("tbl has no param_* columns — check sim_table keys in mass_param.pkl")
else:
    astro = [j for j in range(C.N_PARAMS) if (j+1) not in C.COSMO_PARAM_IDX]

    # per-bin Spearman: truth mass & gen mass vs astro params
    rho_true = {ch: np.full((len(BINS), len(astro)), np.nan) for ch in MASS_CH}
    rho_bind = {ch: np.full((len(BINS), len(astro)), np.nan) for ch in MASS_CH}
    for row, b in enumerate(BINS):
        sub = tbl[tbl.mass_bin == b]
        for ch in MASS_CH:
            t = sub[f'truth_{ch}_rvir']; g = sub[f'gen_{ch}_rvir']
            m = (t > 0) & (g > 0); tv = t[m].values; gv = g[m].values
            for pi, j in enumerate(astro):
                pv = sub[param_cols[j]][m].values
                if pv.std() > 1e-10 and len(tv) > 10:
                    rho_true[ch][row, pi] = spearmanr(tv, pv)[0]
                    rho_bind[ch][row, pi] = spearmanr(gv, pv)[0]

    N_TOP = 6
    cmap_p = plt.cm.tab10
    fig, axes = plt.subplots(len(MASS_CH), 1, figsize=(8, 3.2*len(MASS_CH)), sharex=True)

    for ci, (ch, ax) in enumerate(zip(MASS_CH, axes)):
        # rank params by peak |rho| across bins in truth
        importance = np.nanmax(np.abs(rho_true[ch]), axis=0)
        top_idx = np.argsort(importance)[::-1][:N_TOP]
        for ki, pi in enumerate(top_idx):
            j = astro[pi]; lbl = PARAM_LABELS[j+1]; col = cmap_p(ki / N_TOP)
            ax.plot(range(len(BINS)), rho_true[ch][:, pi], 'o-',  color=col, lw=1.8, ms=5, label=lbl)
            ax.plot(range(len(BINS)), rho_bind[ch][:, pi], 'o--', color=col, lw=1.2, ms=4, alpha=0.6)
        ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.5)
        ax.set_ylim(-1, 1); ax.set_ylabel(f'{CH_DISPLAY[ch]}\n' + r'$\rho_S$')
        ax.legend(fontsize=7, ncol=2, loc='best'); ax.grid(alpha=0.2)

    axes[-1].set_xticks(range(len(BINS))); axes[-1].set_xticklabels([BIN_LABELS[b] for b in BINS], rotation=20)
    axes[-1].set_xlabel(r'$\log_{10} M_{200c}$ bin')
    axes[0].set_title('Top parameter–mass Spearman by bin  (solid = Truth, dashed = BIND)')

    # shared legend for truth vs BIND style
    style_handles = [Line2D([0],[0], color='gray', lw=1.8, ls='-',  label='Truth'),
                     Line2D([0],[0], color='gray', lw=1.2, ls='--', label='BIND', alpha=0.6)]
    axes[0].legend(handles=axes[0].get_legend_handles_labels()[0] + style_handles,
                   labels=axes[0].get_legend_handles_labels()[1] + ['Truth','BIND'],
                   fontsize=7, ncol=3, loc='best')

    save_fig(fig, 'fig3a_param_mass_by_bin'); plt.show()


## §3 · Radial profiles + parameter response

Fig 4 profile fractional residual **by mass bin** (`profiles.pkl`); Fig 3b parameter → profile Spearman (`profiles_r200.pkl`, previously broken — now fixed).

In [ ]:

pr = L('profiles.pkl'); r = pr['r']
fig, axes = plt.subplots(len(MASS_CH), 1, figsize=(7.2, 3.1*len(MASS_CH)), sharex=True)
for c, ax in enumerate(axes):
    for b in BINS:
        if b not in pr['by_bin']: continue
        band = pr['by_bin'][b]
        ax.plot(r, band['pctdiff_med'][c], color=BIN_CMAP[b], lw=1.8, label=BIN_LABELS[b] if c==0 else None)
        ax.fill_between(r, band['pctdiff_p16'][c], band['pctdiff_p84'][c], color=BIN_CMAP[b], alpha=0.10)
    ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.6); ax.set_xscale('log'); ax.set_ylim(-1,1)
    ax.set_ylabel(rf'$\Delta\Sigma_{{\rm {CH_DISPLAY[MASS_CH[c]]}}}/\Sigma$'); ax.grid(which='both', alpha=0.2)
    if c==0: ax.legend(title=r'$\log_{10}M_{200}$', fontsize=8, ncols=2)
axes[-1].set_xlabel(r'$r$ [Mpc$/h$]')
save_fig(fig, 'fig4_radial_pctdiff_by_bin'); plt.show()


In [ ]:

# Fig 3b — parameter -> radial profile Spearman (BIND & residual), FIXED
p2 = L('profiles_r200.pkl'); rr = p2['r_over_r200']; rho = p2['rho_prof']['trained']
astro = [j for j in range(C.N_PARAMS) if (j+1) not in C.COSMO_PARAM_IDX]
order = sorted(astro); xl=[PARAM_LABELS[j+1] for j in order]   # all astro params
r1 = np.argmin(np.abs(rr-1.0))
fig, axes = plt.subplots(len(MASS_CH), 2, figsize=(14, 3.0*len(MASS_CH)), sharex=True, sharey=True, gridspec_kw={'hspace':0.1, 'wspace':0.1})
for c in range(len(MASS_CH)):
    for si,(lab,data) in enumerate([('BIND', rho['BIND'][c][:,order]), ('True − BIND', (rho['True'][c]-rho['BIND'][c])[:,order])]):
        ax = axes[c,si]; im = ax.imshow(data, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5, origin='upper')
        ax.axhline(r1, color='k', ls='--', lw=1, alpha=0.7)
        if si==0: ax.set_ylabel(f'{CH_DISPLAY[MASS_CH[c]]}\n$r/R_{{200}}$')
        if c==0: ax.set_title(lab)
    axes[c,0].set_yticks([np.argmin(np.abs(rr-v)) for v in (0.1,0.5,1,2)]); axes[c,0].set_yticklabels(['0.1','0.5','1','2'])
for ax in axes[-1]: ax.set_xticks(range(len(order))); ax.set_xticklabels(xl, rotation=45, ha='right', fontsize=8)
fig.colorbar(im, ax=axes, fraction=0.12).set_label(r'$\rho_S$')
save_fig(fig, 'fig3b_spearman_param_profile'); plt.show()


## §4 · Field-level response (1P set, Fig 6)

For each feedback parameter, the log-ratio field between its high and low 1P variant (most-massive halo) — truth vs BIND2 (`field1p.npz`).

In [ ]:

f1 = L('field1p.npz'); params = f1['params']
from scipy.stats import pearsonr
nch = 3; nrow = len(params)
fig, axes = plt.subplots(nrow, 2*nch, figsize=(2.0*2*nch, 2.0*nrow))
for ri, j in enumerate(params):
    t_hi,t_lo = f1[f'p{j}_t_hi'], f1[f'p{j}_t_lo']; g_hi,g_lo = f1[f'p{j}_g_hi'], f1[f'p{j}_g_lo']
    for c in range(nch):
        eps = 1e-3*np.percentile(np.r_[t_hi[c][t_hi[c]>0], t_lo[c][t_lo[c]>0]] if (t_hi[c]>0).any() else [1e-6], 99)
        tlr = np.log10((t_hi[c]+eps)/(t_lo[c]+eps)); glr = np.log10((g_hi[c]+eps)/(g_lo[c]+eps))
        v = np.quantile(np.abs(np.r_[tlr.ravel(), glr.ravel()]), 0.995) or 0.1
        for ki,(fld,kind) in enumerate([(tlr,'Truth'),(glr,'BIND')]):
            ax = axes[ri, c*2+ki]; ax.imshow(fld, origin='lower', cmap='RdBu_r', vmin=-v, vmax=v); ax.set_xticks([]); ax.set_yticks([])
            if ri==0: ax.set_title(f'{CH_DISPLAY[MASS_CH[c]]}\n{kind}', fontsize=8)
            if c==0 and ki==0: ax.set_ylabel(PARAM_LABELS[int(j)], fontsize=9, rotation=0, ha='right', va='center', labelpad=22)
            if kind=='BIND':
                s_t,s_g = tlr.std(), glr.std()
                if s_t>1e-9 and s_g>1e-9:
                    ax.text(0.04,0.96,f'r={pearsonr(tlr.ravel(),glr.ravel())[0]:+.2f}\nA={s_g/s_t:.2f}', transform=ax.transAxes, va='top', fontsize=6, bbox=dict(fc='white', ec='none', alpha=0.7))
save_fig(fig, 'fig6_field_response_1p'); plt.show()


## §5 · Full-box power spectrum (Fig 5)

Total-matter $P(k)$ with the corrected **shared-content paste**, trained regime only: BIND ($M_{200c}\ge 10^{13}$) vs truth vs DMO, plus the hydro-replaced control (truth patches of the **same** $\ge 10^{13}$ halos pasted with the same aperture) that isolates model vs aperture error (`pk_fixed.npz`).

In [ ]:

# Diagnostic: inspect pk.npz to understand array shapes and k coverage
_pk = L('pk.npz')
k_diag = _pk['k']
print(f"k range: {k_diag.min():.3f} – {k_diag.max():.3f} h/Mpc  ({len(k_diag)} bins)")
print(f"knyq (1024px, 50 Mpc/h) = {np.pi*1024/50:.2f} h/Mpc")
print()
for s in ('CV', 'Test', '1P', 'SB35'):
    key = f'{s}_bind'
    if key not in _pk: continue
    arr = _pk[key]
    print(f"{s:6s}  bind shape={arr.shape}  finite={np.isfinite(arr).mean():.2f}  "
          f"range=[{np.nanmin(arr):.2e}, {np.nanmax(arr):.2e}]")
    # check for anomalies at high k
    hk = k_diag > 30
    if hk.any():
        t = _pk[f'{s}_truth']
        ratio = arr[:, hk] / t[:, hk]
        print(f"  k>30:  BIND/truth  median={np.nanmedian(ratio):.3f}  "
              f"p16={np.nanquantile(ratio,0.16):.3f}  p84={np.nanquantile(ratio,0.84):.3f}")


In [ ]:

# §5 · Full-box power spectrum — shared-content paste, trained regime (>=1e13) only.
# Reads pk_fixed.npz, built by tools/paper_cache/build_pk_fixed.py:
#     python build_pk_fixed.py --metric fixed --pool 6   # CPU: >=1e13 shared paste + hydro-replace
#     python build_pk_fixed.py --reduce                  # -> pk_fixed.npz

pkf = L('pk_fixed.npz'); k = pkf['k']
knyq = np.pi*1024/50.0
km = k <= knyq; k = k[km]                      # drop super-Nyquist bins
suites = [s for s in ('CV','Test','1P') if f'{s}_fixed_truth' in pkf]

def med(a):  return np.median(a[:, km], 0)
def band(num, den):
    r = num[:, km] / den[:, km]
    return np.median(r, 0), np.quantile(r, 0.16, 0), np.quantile(r, 0.84, 0)

fig, axes = plt.subplots(2, len(suites), figsize=(5.2*len(suites), 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1], 'hspace': 0}, squeeze=False)
for col, s in enumerate(suites):
    at, ab = axes[0, col], axes[1, col]
    tr, dm = pkf[f'{s}_fixed_truth'], pkf[f'{s}_fixed_dmo']
    bind = pkf[f'{s}_fixed_ge13']
    fin = np.isfinite(bind).all(1)             # sims lacking >=1e13 halos are NaN rows

    at.plot(k, med(tr)/med(dm), 'k', lw=1.4, label='Truth/DMO')
    at.plot(k, med(bind[fin])/med(dm[fin]), color='tab:orange', lw=1.4,
            label=r'BIND ($M_{200c}\geq 10^{13}$)/DMO')
    rm, rlo, rhi = band(bind[fin], tr[fin])
    ab.plot(k, rm, color='tab:orange', lw=1.3, label='BIND/Truth')
    ab.fill_between(k, rlo, rhi, color='tab:orange', alpha=0.16)

    # hydro-replaced control: truth patches of the SAME >=1e13 halos, same shared
    # paste (isolates model error from paste/aperture error)
    hr, tr_hr = pkf.get(f'{s}_fixed_hr_ge13'), tr
    if hr is None or not np.isfinite(hr).any():
        # cache predates hr_ge13 -> fall back to the legacy all-catalog-halo control
        hr, tr_hr = pkf.get(f'{s}_pkfile_hydro_replace'), pkf.get(f'{s}_pkfile_truth', tr)
        if hr is not None and col == 0:
            print('WARNING: pk_fixed.npz lacks hr_ge13 — showing the legacy (>=1e12) '
                  'hydro-replace; rerun build_pk_fixed.py --metric fixed then --reduce')
    if hr is not None and np.isfinite(hr).any():
        fh = np.isfinite(hr).all(1)
        at.plot(k, med(hr[fh])/med(dm[fh]), color='tab:blue', lw=1.2, label='Hydro-repl/DMO')
        ab.plot(k, np.median(hr[fh][:, km]/tr_hr[fh][:, km], 0), color='tab:blue', ls='-.',
                lw=1, label='Hydro-repl/Truth')
    ab.plot(k, np.median(dm[:, km]/tr[:, km], 0), 'gray', ls='--', lw=1, label='DMO/Truth')

    at.axhline(1, color='gray', lw=0.6, ls='--'); at.set_xscale('log')
    at.grid(which='both', alpha=0.25); at.set_title(SUITE_DISPLAY[s]); at.tick_params(labelbottom=False)
    ab.axhspan(0.8, 1.2, color='tab:green', alpha=0.08); ab.axhline(1, color='k', lw=0.6, ls='--')
    ab.set_ylim(0.5, 1.5); ab.set_xscale('log'); ab.grid(which='both', alpha=0.25)
    ab.set_xlabel(r'$k$ [$h$/Mpc]')
    if col == 0:
        at.set_ylabel(r'$P/P_{\rm DMO}$'); at.legend(fontsize=8)
        ab.set_ylabel(r'$P/P_{\rm Truth}$'); ab.legend(fontsize=7, loc='lower left')

save_fig(fig, 'fig5_total_field_pk_fixed'); plt.show()


## §6 · Halo shapes + parameter response (Fig 7)

Mass-weighted 2D axis-ratio $q$ distributions (truth vs BIND vs DMO) and the **shape parameter response** (`shapes.pkl`).

In [ ]:

# §6 intro — example halo patches + mass-weighted quadrupole ellipse illustration
# Shows one halo near each of 10^14.5, 10^14, 10^13.5, 10^13 M_sun.

_rec0   = C.discover_sims(('CV',))[0]
_fm0    = C.load_full_maps(_rec0)
_cat0   = C.load_catalog(_rec0)
_gen0   = C.load_generated(_rec0)
_cpx0   = C.centers_to_pixels(_cat0['centers'])
_truth0 = C.extract_truth_mass_patches(_fm0, _cpx0)     # (N, 3, 128, 128)
_gen0m  = _gen0[:, :C.N_MASS_CH]                         # (N, 3, 128, 128)
_dmo0   = np.stack([C.extract_patch(_fm0['dmo_fullbox'], cx, cy) for cx, cy in _cpx0])
_r200_0 = C.r200_pix_patch(_cat0)
_masses0 = np.log10(np.asarray(_cat0['masses']))

# pick one halo closest to each target mass (require r200 > 4 px)
_valid0 = np.where(np.isfinite(_r200_0) & (_r200_0 > 4))[0]
_targets = [14.5, 14.0, 13.5, 13.0]
_picks = np.array([
    _valid0[np.argmin(np.abs(_masses0[_valid0] - t))]
    for t in _targets
])
print("Selected halo log-masses:", _masses0[_picks])

def _quad_shape(img, r200, thr=0.0):
    """Mass-weighted quadrupole axis ratio q and position angle pa [rad]."""
    H, W = img.shape
    yy, xx = np.mgrid[0:H, 0:W].astype(float)
    ap = (xx - (W - 1) / 2) ** 2 + (yy - (H - 1) / 2) ** 2 <= r200 ** 2
    w = np.maximum(img.astype(float) - thr, 0.0) * ap
    tot = w.sum()
    if tot < 1e-30 or int((w > 0).sum()) < 5:
        return np.nan, np.nan
    xc = (xx * w).sum() / tot;  yc = (yy * w).sum() / tot
    dx, dy = xx - xc, yy - yc
    Qxx = (dx ** 2 * w).sum() / tot
    Qyy = (dy ** 2 * w).sum() / tot
    Qxy = (dx * dy * w).sum() / tot
    evals, evecs = np.linalg.eigh([[Qxx, Qxy], [Qxy, Qyy]])
    lmin, lmax = evals[0], evals[1]
    if lmax < 1e-30 or lmin < 0:
        return np.nan, np.nan
    return np.sqrt(lmin / lmax), np.arctan2(evecs[1, 1], evecs[0, 1])

from matplotlib.patches import Ellipse as _Ell, Circle as _Circ

_ROWS       = 4
_row_labels = ['DMO', 'DM (hydro)', 'Gas', 'Stars']
_row_cmaps  = ['gray_r', 'Blues', 'Greens', 'Purples']
_row_thrs   = [None, 0.0, 0.0, C.STAR_THRESH]   # None → skip ellipse (DMO)

N_SHOW = len(_picks)
fig, axes = plt.subplots(_ROWS, N_SHOW, figsize=(2.5 * N_SHOW, 2.5 * _ROWS),
                         gridspec_kw={'hspace': 0.04, 'wspace': 0.04})

for col, hi in enumerate(_picks):
    r200 = float(_r200_0[hi])
    _imgs = [_dmo0[hi], _truth0[hi, 0], _truth0[hi, 1], _truth0[hi, 2]]
    for row in range(_ROWS):
        ax = axes[row, col]
        img = _imgs[row]
        pos = img[img > 0]
        if len(pos):
            lo_v = np.quantile(pos, 0.05)
            hi_v = np.quantile(pos, 0.999)
            ax.imshow(np.log10(np.clip(img, lo_v, None)), origin='lower',
                      cmap=_row_cmaps[row], interpolation='nearest',
                      vmin=np.log10(lo_v), vmax=np.log10(hi_v))
        else:
            ax.imshow(np.zeros_like(img), origin='lower', cmap=_row_cmaps[row])
        ax.set_xticks([]); ax.set_yticks([])

        # R200c aperture circle
        ax.add_patch(_Circ((63.5, 63.5), r200, color='lime',
                           fill=False, lw=1.2, ls='--', zorder=3))

        # quadrupole shape ellipse (hydro channels only)
        if _row_thrs[row] is not None:
            q_v, pa_v = _quad_shape(img, r200, thr=_row_thrs[row])
            if np.isfinite(q_v):
                ax.add_patch(_Ell((63.5, 63.5),
                                  width=2 * r200, height=2 * r200 * q_v,
                                  angle=np.degrees(pa_v),
                                  edgecolor='white', facecolor='none',
                                  lw=1.8, zorder=4))
                ax.text(0.97, 0.03, f'$q={q_v:.2f}$',
                        transform=ax.transAxes, ha='right', va='bottom',
                        fontsize=7.5, color='white',
                        bbox=dict(facecolor='k', alpha=0.35, pad=1.5, boxstyle='round,pad=0.2'))

        if col == 0:
            ax.set_ylabel(_row_labels[row], fontsize=9)
        if row == 0:
            ax.set_title(rf'$\log M={_masses0[hi]:.1f}$', fontsize=8.5)

# legend
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
_leg = [
    Line2D([0], [0], color='lime', ls='--', lw=1.4,
           label=r'$R_{200c}$ aperture'),
    Line2D([0], [0], color='gray', lw=1.8,
           label=r'Quadrupole ellipse  ($q = \sqrt{\lambda_\mathrm{min}/\lambda_\mathrm{max}}$)'),
]
fig.legend(handles=_leg, loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.03), framealpha=0.85)
fig.suptitle('Halo patches and mass-weighted quadrupole shape measurement', fontsize=11, y=1.01)
save_fig(fig, 'fig6_shape_illustration'); plt.show()


In [ ]:

sh = L('shapes.pkl'); A = sh['arrays']; suite = A['suite']
fig, axes = plt.subplots(3, 1, figsize=(6.5, 9), sharex=True)
bins = np.linspace(0,1,31)
for row,(key,name) in enumerate([('dm','DM'),('gas','Gas'),('star','Stars')]):
    ax = axes[row]
    for src,ls,lab in [('truth','-','Truth'),('gen','--','BIND')]:
        q = A[f'{src}_{key}_q']; q=q[np.isfinite(q)]
        ax.hist(q, bins=bins, density=True, histtype='stepfilled' if src=='truth' else 'step', alpha=0.4 if src=='truth' else 1, lw=2, color='tab:gray' if src=='truth' else 'tab:orange', label=lab)
    if key=='dm':
        qd=A['dmo_q']; qd=qd[np.isfinite(qd)]; ax.hist(qd, bins=bins, density=True, histtype='step', ls=':', color='k', label='DMO')
    ax.set_ylabel(rf'$\rho_{{\rm {name}}}(q)$');
    if row==0: ax.legend()
axes[-1].set_xlabel('$q$ (minor/major axis ratio)')
save_fig(fig, 'fig7_shape_axisratio'); plt.show()


In [ ]:

# shape parameter response — BIND (gen) axis-ratio q vs params
rho = sh['rho_shape']['trained']; mets = list(sh['response_metrics'])
gen_rows = [i for i,m in enumerate(mets) if m.startswith('gen_') and 'eps' not in m]
astro = [j for j in range(C.N_PARAMS) if (j+1) not in C.COSMO_PARAM_IDX]
order = sorted(astro); xl=[PARAM_LABELS[j+1] for j in order]
fig, ax = plt.subplots(figsize=(12, 3.4))
d = rho[np.ix_(gen_rows, order)]; im = ax.imshow(d, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax.set_yticks(range(len(gen_rows))); ax.set_yticklabels([mets[i].replace('gen_','') for i in gen_rows])
ax.set_xticks(range(len(order))); ax.set_xticklabels(xl, rotation=45, ha='right')
for i in range(len(gen_rows)):
    for kk in range(len(order)):
        if np.isfinite(d[i,kk]): ax.text(kk,i,f'{d[i,kk]:.2f}',ha='center',va='center',fontsize=7,color='white' if abs(d[i,kk])>0.4 else 'k')
fig.colorbar(im, ax=ax, fraction=0.02).set_label(r'$\rho_S$'); ax.set_title('BIND shape response to parameters')
save_fig(fig, 'fig7b_shape_param_response'); plt.show()
